
# 00 · Project setup and replication environment

This notebook initializes the project structure for the geospatial causal workflow.

## What this notebook does
1. Detects or sets the project root.
2. Creates the folder structure used by the workflow.
3. Records the expected Python and package versions for replication.
4. Writes a project configuration file to `configs/project_config.json`.
5. Writes an environment manifest to `configs/environment_manifest.json`.
6. Applies a simple plotting theme that later notebooks can reuse.

## Replication principle
This notebook is deliberately **self-contained**. It does not import local helper modules.
Every helper needed at this stage is defined inside the notebook.


In [ ]:

# Optional installation cell for exact replication in the CURRENT notebook environment.
# Run this only if you need to install or update packages.

from pathlib import Path
import sys

req = (Path.cwd().resolve().parent / "requirements.txt").resolve()
print("Using requirements file:", req)
print("Current Python executable:", sys.executable)


Using requirements file: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\requirements.txt
Current Python executable: c:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\.venv\Scripts\python.exe


In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:

from pathlib import Path
import importlib
import importlib.metadata as ilm
import json
import platform
import sys
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



## Detect the project root

The notebook first tries to infer the project root from the current working directory.
If that fails in your local setup, set `MANUAL_PROJECT_ROOT` to the absolute path of the project folder.


In [31]:

MANUAL_PROJECT_ROOT = None
# Example for Windows:
# MANUAL_PROJECT_ROOT = r"C:\Users\YOUR_NAME\OneDrive - Hertie School\PhD\Paper 2\paper2"

def detect_project_root(manual_root=None):
    if manual_root is not None:
        return Path(manual_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
    return cwd

PROJECT_ROOT = detect_project_root(MANUAL_PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd().resolve())


Project root: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2
Current working directory: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\notebooks


## Helper functions

In [32]:

def ensure_project_structure(project_root: Path):
    folders = [
        project_root / "notebooks",
        project_root / "configs",
        project_root / "data" / "raw",
        project_root / "data" / "intermediate",
        project_root / "data" / "processed",
        project_root / "outputs" / "figures",
        project_root / "outputs" / "maps",
        project_root / "outputs" / "tables",
        project_root / "logs",
    ]
    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)
    return folders


def print_tree(path: Path, max_depth: int = 3, prefix: str = "", _depth: int = 0):
    if _depth == 0:
        print(path.name + "/")
    if _depth >= max_depth:
        return

    entries = sorted(list(path.iterdir()), key=lambda p: (p.is_file(), p.name.lower()))
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name + ("/" if entry.is_dir() else ""))
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, max_depth=max_depth, prefix=prefix + extension, _depth=_depth + 1)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def set_plot_theme():
    plt.style.use("default")
    plt.rcParams.update({
        "figure.figsize": (10, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "figure.dpi": 120,
        "savefig.bbox": "tight",
    })


def collect_environment_manifest(packages):
    manifest = {
        "timestamp_utc": datetime.now(UTC).isoformat().replace("+00:00", "Z"),
        "python": sys.version,
        "platform": platform.platform(),
        "packages": {},
    }
    for pkg in packages:
        try:
            manifest["packages"][pkg] = ilm.version(pkg)
        except Exception:
            manifest["packages"][pkg] = None
    return manifest


## Create the folder structure and save the environment manifest

In [33]:

ensure_project_structure(PROJECT_ROOT)
set_plot_theme()

EXPECTED_ENVIRONMENT = {
    "python": "3.13+",
    "packages": {
        "numpy": ">=2.1.0",
        "scipy": ">=1.10.0",
        "pandas": ">=2.2.3",
        "pyarrow": ">=18.0.0",
        "matplotlib": ">=3.9.0",
        "geopandas": ">=1.0.1",
        "shapely": ">=2.0.6",
        "pyproj": ">=3.7.0",
        "pyogrio": ">=0.10.0",
        "earthengine-api": ">=1.6.0",
        "geemap": ">=0.35.0",
        "ipykernel": ">=6.29.5",
        "jupyterlab": ">=4.4.0",
    },
}

project_config_path = PROJECT_ROOT / "configs" / "project_config.json"
if project_config_path.exists():
    with open(project_config_path, "r", encoding="utf-8") as f:
        existing_project_config = json.load(f)
else:
    existing_project_config = {}

project_config = existing_project_config.copy()
project_config.update({
    "project_name": "geospatial_causal_workflow",
    "created_from_notebook": "00_project_setup.ipynb",
    "project_root": str(PROJECT_ROOT),
    "notes": "Notebooks 01-03 cover panel construction, treatment timing, and spatial structure; notebook 03 requires scipy for neighbor search.",
})
project_config["data_dirs"] = {
    **existing_project_config.get("data_dirs", {}),
    "raw": str(PROJECT_ROOT / "data" / "raw"),
    "intermediate": str(PROJECT_ROOT / "data" / "intermediate"),
    "processed": str(PROJECT_ROOT / "data" / "processed"),
}
project_config["output_dirs"] = {
    **existing_project_config.get("output_dirs", {}),
    "figures": str(PROJECT_ROOT / "outputs" / "figures"),
    "maps": str(PROJECT_ROOT / "outputs" / "maps"),
    "tables": str(PROJECT_ROOT / "outputs" / "tables"),
}

manifest = collect_environment_manifest([
    "numpy", "scipy", "pandas", "pyarrow", "matplotlib", "geopandas", "shapely", "pyproj", "pyogrio", "earthengine-api", "geemap", "ipykernel", "jupyterlab"
])

save_json(project_config, project_config_path)
save_json(EXPECTED_ENVIRONMENT, PROJECT_ROOT / "configs" / "expected_environment.json")
save_json(manifest, PROJECT_ROOT / "configs" / "environment_manifest.json")

print("Saved:")
print(" -", project_config_path)
print(" -", PROJECT_ROOT / "configs" / "expected_environment.json")
print(" -", PROJECT_ROOT / "configs" / "environment_manifest.json")
print()
print_tree(PROJECT_ROOT, max_depth=3)


Saved:
 - C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\configs\project_config.json
 - C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\configs\expected_environment.json
 - C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\configs\environment_manifest.json

paper2/
├── .git/
│   ├── hooks/
│   │   ├── applypatch-msg.sample
│   │   ├── commit-msg.sample
│   │   ├── fsmonitor-watchman.sample
│   │   ├── post-update.sample
│   │   ├── pre-applypatch.sample
│   │   ├── pre-commit.sample
│   │   ├── pre-merge-commit.sample
│   │   ├── pre-push.sample
│   │   ├── pre-rebase.sample
│   │   ├── pre-receive.sample
│   │   ├── prepare-commit-msg.sample
│   │   ├── push-to-checkout.sample
│   │   ├── sendemail-validate.sample
│   │   └── update.sample
│   ├── info/
│   │   └── exclude
│   ├── logs/
│   │   ├── refs/
│   │   └── HEAD
│   ├── objects/
│   │   ├── info/
│   │   └── pack/
│   ├── refs/
│   │   ├── heads/
│   │   ├── remotes/
│   │   └── tags/
│   ├── c

C:\Users\cpedr\AppData\Local\Temp\ipykernel_38128\2555317685.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat() + "Z",


## Quick version report

In [ ]:

print("Python:", sys.version)
package_import_map = {
    "numpy": "numpy",
    "scipy": "scipy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "matplotlib": "matplotlib",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "pyproj": "pyproj",
    "pyogrio": "pyogrio",
    "earthengine-api": "ee",
    "geemap": "geemap",
    "ipykernel": "ipykernel",
    "jupyterlab": "jupyterlab",
}
for pkg, import_name in package_import_map.items():
    try:
        importlib.import_module(import_name)
        print(f"{pkg}: {ilm.version(pkg)}")
    except Exception as exc:
        print(f"{pkg}: not installed or failed to import ({exc})")


Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
numpy: 2.4.4
pandas: 2.3.3
pyarrow: 23.0.1
matplotlib: 3.10.8
geopandas: 1.1.3
shapely: 2.1.2
pyproj: 3.7.2
pyogrio: 0.12.1
earthengine-api: 1.7.19
geemap: 0.37.2
ipykernel: 7.2.0
jupyterlab: 4.5.6



## Expected outputs from this notebook

After running this notebook, you should have:
- a reproducible folder structure,
- a project configuration JSON,
- an expected environment JSON,
- and an environment manifest capturing the versions actually installed in the current kernel.

The next notebooks use this structure to build the grid panel, define treatment timing, and then construct the spatial backbone used for later spillover analysis.
